In [2]:
import os
from pathlib import Path

from f_dirs import get_data_dirs
dirs = get_data_dirs()

# Iterate over the attributes of the DirPaths object
for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

data_dir: H:\Other computers\My computer\fame_clean\1_FAME_raw_data\2025.07.30
output_dir: C:\Users\lazym\Documents\Code\dissertation\build\output
raw_data_dir: H:\Other computers\My computer\fame_clean\1_FAME_raw_data\2025.02
root_data_dir: H:\Other computers\My computer\fame_clean
root_dir: C:\Users\lazym\Documents\Code\dissertation
work_dir: C:\Users\lazym\Documents\Code\dissertation\build\src


### 1. Generate the raw file dictionary

This script resolves the project data paths, scans the raw-data folder, builds a nested dictionary of file metadata by company and file category, and writes it to JSON.  
`get_data_dirs()` defines the working directories  
`build_raw_file_dict()` performs the recursive traversal and file collection through its helper functions.  

In [4]:
from flask import json
import pandas as pd

from f_traverse import build_raw_file_dict
pd.options.mode.chained_assignment = None  # default='warn'

if dirs.raw_data_dir is None:
	raise ValueError("raw_data_dir is None. Please check your .env file and ensure RAW_DATA_DIR is set correctly.")

raw_file_dict = build_raw_file_dict(dirs.raw_data_dir)
with open(dirs.output_dir / "raw_file_dict.json", "w") as f:
    json.dump(raw_file_dict, f, indent=4)
    print(f"✅ Successfully built raw file dictionary and saved to: {dirs.output_dir / 'raw_file_dict.json'}")

Traversing industry directory: 8̶8̶,̶ ̶5̶2̶,̶ ̶6̶1̶,̶ ̶9̶4̶,̶ ̶4̶7̶,̶ ̶4̶6̶,̶ ̶5̶6̶,̶ ̶9̶0̶,̶ ̶3̶2̶,̶ ̶2̶3̶,̶ ̶4̶3̶,̶ ̶6̶2̶,̶ ̶8̶2̶,̶ ̶6̶8̶,̶ ̶7̶0̶, 86

KeyboardInterrupt: 

### 2. Ensure database connection

In [33]:
import pandas as pd
# Import ibis-framework
import ibis
# pip install 'ibis-framework[duckdb,examples]' # examples is only required to access the sample data Ibis provides

print("Path:", ibis.__file__)
print("Version:", getattr(ibis, "__version__", "No version found"))
pd.options.mode.chained_assignment = None  # default='warn'

db_path = dirs.output_dir / "fame_data.duckdb"

# import xlsx file from input/raw_properties.xlsx to load as a schema
# Declare types that the schema_raw_obj df has colums ["from_raw", "key", "type", "fuzzy_mapping", "in_raw_data", "keep", "in_ln_set", "description"]
schema_path = dirs.root_dir / "build" / "input" / "raw_properties.xlsx"
schema_raw_obj = pd.read_excel(schema_path, sheet_name="raw_properties", engine="openpyxl")

# Fixed schema
schema_fixed = schema_raw_obj[schema_raw_obj["from_raw"] == "a1_ID"]
schema_fixed_dict = dict(zip(schema_fixed["key"], schema_fixed["type"]))
schema_fixed_ibis = ibis.schema(schema_fixed_dict)
schema_fixed_names = set(schema_fixed_ibis.names)
assert len(schema_fixed_ibis.names) == 49, f"Expected 49 columns in schema_fixed_ibis, but found {len(schema_fixed_ibis.names)}"

schema_derived = schema_raw_obj[schema_raw_obj["from_raw"] == "derived"]

# 4. Execute the table creation using the Ibis schema
try:
    
    # 2. Connect to DuckDB using Ibis
    con = ibis.duckdb.connect(str(db_path))
    print(f"Initializing DuckDB via Ibis at: {db_path}")
    # overwrite=True prevents errors if the script is run multiple times during setup
    con.create_table("fame_fixed", schema=schema_fixed_ibis, overwrite=True)
    print("✅ Successfully created Ibis schema for 'fame_fixed'.")
    print("\nTable Schema Verification:")
    print(con.table("fame_fixed").schema())
    
except Exception as e:
    print(f"❌ Error creating table: {e}")

Path: c:\Users\lazym\Documents\Code\dissertation\.venv\Lib\site-packages\ibis\__init__.py
Version: 12.0.0
Initializing DuckDB via Ibis at: C:\Users\lazym\Documents\Code\dissertation\build\output\fame_data.duckdb
❌ Error creating table: No module named 'pyarrow'


# 3. Process each excel file in the raw file dictionary

In [42]:
# Create a dict fuzzy matching the scheme to the following column names in the raw data
# Raw column name -> FAME schema name mapping, based on every possibility of the raw column names
schema_fixed_fuzzy_df = schema_fixed[["key", "fuzzy_mapping"]]
schema_fixed_fuzzy_df["fml"] = schema_fixed_fuzzy_df["fuzzy_mapping"].str.split('\n')
fuzzy_col_mapping = dict(zip(schema_fixed_fuzzy_df["key"], schema_fixed_fuzzy_df["fml"]))
fuzzy_col_to_schema_mapping = {
    raw_name: schema_name for schema_name,
    raw_names in fuzzy_col_mapping.items() for raw_name in raw_names
}
# with open(dirs.root_dir / "build" / "tmp" / "fuzzy_col_mapping.json", "w") as f:
#     json.dump(fuzzy_col_mapping, f, indent=4)
#     print(f"✅ Successfully saved fuzzy column mapping to: {dirs.output_dir / 'fuzzy_col_mapping.json'}")

def check_df_matches_schema(df: pd.DataFrame, test_mode: bool = False) -> bool:

    # For each column in the DataFrame, check if the column name exists in the fame.schema
    # And tick off every column in fame.schema that is found
    # if any are missing from the given DataFrame, then throw an error
    df_column_names = set(df.columns)
    missing_columns = schema_fixed_names - df_column_names
    extra_columns = df_column_names - schema_fixed_names

    # Check for missing or extra columns
    if missing_columns:
        if not test_mode:
            raise ValueError(f"❌ Missing columns in DataFrame: {missing_columns}")
        return False

    if extra_columns:
        if not test_mode:
            raise ValueError(f"❌ Extra columns in DataFrame: {extra_columns}")
        return False

    # Check if the number of columns matches
    if len(df.columns) != len(schema_fixed_names):
        if not test_mode:
            raise ValueError(f"❌ Column count mismatch: Expected {len(schema_fixed_names)} columns, but got {len(df.columns)}")
        return False
    
    return True

# Tests for fuzzy_col_to_schema_mapping
def test_fuzzy_col_to_schema_mapping():
    # Test that all raw names map to the correct schema name
    for schema_name, raw_names in fuzzy_col_mapping.items():
        for raw_name in raw_names:
            assert fuzzy_col_to_schema_mapping[raw_name] == schema_name, f"Test failed for raw name '{raw_name}' mapping to schema name '{schema_name}'"
    
    # Test a few hardcoded examples
    assert fuzzy_col_to_schema_mapping["Company name"] == "company_name", "Test failed for 'Company name'"
    assert fuzzy_col_to_schema_mapping["R/O Address, line 1"] == "ro_address_line_1", "Test failed for 'R/O Address, line 1'"
    assert fuzzy_col_to_schema_mapping["Primary trading address Latitude"] == "primary_trading_address_latitude", "Test failed for 'Primary trading address Latitude'"
    assert fuzzy_col_to_schema_mapping["Latest accounts date"] == "latest_accounts_date", "Test failed for 'Latest accounts date'"
    
    print("✅ All tests passed for fuzzy_col_to_schema_mapping.")

# Test for check_df_matches_schema based on the ibis.schema object
# don't use the fuzzy_col_by_index object because it doesn't contain all the columns of the schema
def test_check_df_matches_schema():

    # Create a DataFrame with correct column names
    correct_df = pd.DataFrame(columns=schema_fixed_ibis.names)
    assert check_df_matches_schema(correct_df) == True, "Test failed for DataFrame with correct schema"
    
    # Create a DataFrame with missing columns
    incorrect_df_missing = pd.DataFrame(columns=schema_fixed_ibis.names[:-1])  # Remove last column
    assert check_df_matches_schema(incorrect_df_missing, test_mode=True) == False, "Test failed for DataFrame with missing columns"
    
    # Create a DataFrame with extra columns
    incorrect_df_extra = pd.DataFrame(columns=list(schema_fixed_ibis.names) + ["extra_column"])
    assert check_df_matches_schema(incorrect_df_extra, test_mode=True) == False, "Test failed for DataFrame with extra columns"
    
    # Create a DataFrame with incorrect column count (fewer columns)
    incorrect_df_count = pd.DataFrame(columns=[col for col in schema_fixed_ibis.names[:-1]])  # Remove last column
    assert check_df_matches_schema(incorrect_df_count, test_mode=True) == False, "Test failed for DataFrame with incorrect column count"

    # Hardcode a test for all the column names in the scheme in a dataframe and check that the function returns True
    hard_names = [
                    "company_name", "registered_number", "ticker_symbol", "ro_address", "ro_address_line_1", "ro_address_line_2", "ro_address_line_3",
                    "ro_address_line_4", "ro_address_line_5", "ro_city", "ro_county", "ro_postcode", "ro_full_postcode", "ro_country", "ro_latitude", "ro_longitude",
                    "ro_nuts_region", "ro_postal_region", "ro_phone", "ro_phone_registered_on_tps", "ro_phone_registered_on_ctps", "primary_trading_address", "primary_trading_address_latitude",
                    "primary_trading_address_longitude", "primary_trading_address_no_of_employees", "branch_name", "trade_description", "primary_uk_sic_2007_code",
                    "primary_uk_sic_2007_description", "full_overview", "history", "primary_business_line", "secondary_business_line", "main_activity", "secondary_activity",
                    "main_products_and_services", "size_estimate", "strategy_organization_and_policy", "strategic_alliances", "membership_of_a_network", "main_brand_names",
                    "main_domestic_country", "main_foreign_countries_or_regions", "main_production_sites", "main_distribution_sites", "main_sales_representation_sites", "main_customers"
                    , "latest_accounts_date", "no_of_available_years"
                ]
    hardcoded_df = pd.DataFrame(columns=hard_names)
    assert check_df_matches_schema(hardcoded_df) == True, "Test failed for hardcoded DataFrame with correct schema"

    hard_names2 = hard_names + ["extra_column"]
    hardcoded_df2 = pd.DataFrame(columns=hard_names2)
    assert check_df_matches_schema(hardcoded_df2, test_mode=True) == False, "Test should have rejected hardcoded DataFrame with extra column"

    hard_names3 = hard_names[:-1]
    hardcoded_df3 = pd.DataFrame(columns=hard_names3)
    assert check_df_matches_schema(hardcoded_df3, test_mode=True) == False, "Test failed for hardcoded DataFrame with missing column"

    print("✅ All tests passed for check_df_matches_schema.")

test_fuzzy_col_to_schema_mapping()
test_check_df_matches_schema()

✅ All tests passed for fuzzy_col_to_schema_mapping.
✅ All tests passed for check_df_matches_schema.


In [43]:
# Set flags in the DataFrame to indicate whether data was available for certain columns.
def set_data_av_flags(df: pd.DataFrame) -> pd.DataFrame:

    # Create a whole new column in the dataframe which has a value based on the other columns
    # If there is no primary trading address, then the value is False
    # If there is a primary trading adress and it is notna, then the value is True
    if 'primary_trading_address' not in df.columns:
        df['data_av_ptaddress'] = False
    else:
        df['data_av_ptaddress'] = df['primary_trading_address'].notna()
    if 'primary_trading_address_latitude' not in df.columns or 'primary_trading_address_longitude' not in df.columns:
        df['data_av_ptaddress_latlong'] = False
    else:
        df['data_av_ptaddress_latlong'] = df['primary_trading_address_latitude'].notna() & df['primary_trading_address_longitude'].notna()

    return df

def test_set_data_av_flags():
    test_df1 = pd.DataFrame({
        "primary_trading_address": ["123 Main St", None, "456 Elm St"],
        "primary_trading_address_latitude": [51.5074, None, 40.7128],
        "primary_trading_address_longitude": [-0.1278, None , -74.0060]
    })
    result_df1 = set_data_av_flags(test_df1)
    assert result_df1['data_av_ptaddress'].tolist() == [True, False, True], "Test failed for data_av_ptaddress flag"
    test_df2 = pd.DataFrame({
        "primary_trading_address": [None, None, None],
        "primary_trading_address_latitude": [None, None, None],
        "primary_trading_address_longitude": [None, None, None]
    })
    result_df2 = set_data_av_flags(test_df2)
    assert result_df2['data_av_ptaddress'].tolist() == [False, False, False], "Test failed for data_av_ptaddress flag with all None"
    assert result_df2['data_av_ptaddress_latlong'].tolist() == [False, False, False], "Test failed for data_av_ptaddress_latlong flag with all None"
    test_df3 = pd.DataFrame({
        "primary_trading_address": ["123 Main St", "456 Elm St", "789 Oak St"]
    })
    result_df3 = set_data_av_flags(test_df3)
    assert result_df3['data_av_ptaddress'].tolist() == [True, True, True], "Test failed for data_av_ptaddress flag with only primary_trading_address column"
    assert result_df3['data_av_ptaddress_latlong'].tolist() == [False, False, False], "Test failed for data_av_ptaddress_latlong flag with only primary_trading_address column"
    test_df4 = pd.DataFrame({
        "primary_trading_address_latitude": [51.5074, 40.7128, 34.0522],
        "primary_trading_address_longitude": [-0.1278, -74.0060, -118.2437]
    })
    result_df4 = set_data_av_flags(test_df4)
    assert result_df4['data_av_ptaddress'].tolist() == [False, False, False], "Test failed for data_av_ptaddress flag with only latitude and longitude columns"
    assert result_df4['data_av_ptaddress_latlong'].tolist() == [True, True, True], "Test failed for data_av_ptaddress_latlong flag with only latitude and longitude columns"
    print("✅ All tests passed for set_data_av_flags function.")

### --- ###

test_set_data_av_flags()


✅ All tests passed for set_data_av_flags function.


In [ ]:
from flask import json
import random

# 3. Process each excel file in the raw file dictionary
# Extract the data from each Excel file from tab 'Results'
# Where headers are in row 1 and rows are below that
# And the first column 'Company name' is column B
# Enter the data into the above fame_schema and console.log it

# Traverse the raw_file_dict.json file to get each Excel filepath
# declare raw_file_dict as a RawFileDict type
raw_file_dict: RawFileDict | None = None

with open(dirs.output_dir / "raw_file_dict.json", "r") as f:
    raw_file_dict = json.load(f)

if raw_file_dict is None:
    raise ValueError("❌ Error: raw_file_dict.json is empty or not found.")

process_count = 0

# We want one big dataframe, which we will merge all the data into for now
df_fixed = pd.DataFrame(columns=list(fame_schema.names))
ind_keys = raw_file_dict.keys()
ind_shuffled = list(ind_keys)
random.shuffle(ind_shuffled)

for ind in ind_shuffled:

    obj = raw_file_dict[ind]

    for property, arr in obj.items():

        if process_count >= 5:
            break

        if property != "a1_ID":
            continue

        files_shuffled = arr.copy()
        random.shuffle(files_shuffled)
        for f in files_shuffled:
            [file_name, file_path] = f

            try:

                # Read the Excel file, assuming the relevant data is in the 'Results' sheet
                # Drop column A
                # For each raw colum name, map it to the fame_schema name using fuzzy_col_to_schema_mapping
                # Set the data availability flags
                file_df = pd.read_excel(file_path, sheet_name='Results', header=0)
                file_df.drop(file_df.columns[0], axis=1, inplace=True)
                file_df = drop_duplicate_columns(file_df)
                file_df.rename(columns=fuzzy_col_to_schema_mapping, inplace=True)
                flagged_df = set_data_av_flags(file_df)
                # If columns are missing from the df, add them with NaN values
                for col in fame_schema.names:
                    if col not in flagged_df.columns:
                        flagged_df[col] = pd.NA
                check_df_matches_schema(flagged_df)

                # Merge the current file_df into the main df
                df_fixed = pd.concat([df_fixed, flagged_df], ignore_index=True)

                # # Insert data into DuckDB table
                # con.insert("fame_id_data", file_df)
                print(f"✅ Successfully processed and inserted data from: {ind}/{property}/{file_name}")
                
            except Exception as e:
                print(f"❌ Error processing file {ind}/{property}/{file_name}")
                print("Error:", e)
                print("\n")

        process_count += 1

# Output the df to a CSV file for inspection
output_csv_path = dirs.output_dir / "merged_fame_data.csv"
df_fixed.to_csv(output_csv_path, index=False)
print(f"✅ Merged DataFrame saved to: {output_csv_path}")

✅ Successfully processed and inserted data from: 99/a1_ID/Export 15_02_2025 12_49.xlsx
✅ Successfully processed and inserted data from: 87/a1_ID/Export 20_02_2025 19_23.xlsx
✅ Successfully processed and inserted data from: 87/a1_ID/Export 20_02_2025 19_22 1.xlsx
✅ Successfully processed and inserted data from: 87/a1_ID/Export 20_02_2025 19_22 2.xlsx
✅ Successfully processed and inserted data from: 87/a1_ID/Export 20_02_2025 19_22.xlsx
✅ Successfully processed and inserted data from: 87/a1_ID/Export 20_02_2025 19_23 1.xlsx
✅ Successfully processed and inserted data from: 14/a1_ID/Export 15_02_2025 21_56 1.xlsx
✅ Successfully processed and inserted data from: 14/a1_ID/Export 15_02_2025 21_56.xlsx
✅ Successfully processed and inserted data from: 23/a1_ID/Export 15_02_2025 22_17.xlsx
✅ Successfully processed and inserted data from: 21/a1_ID/Export 15_02_2025 22_16.xlsx
✅ Merged DataFrame saved to: C:\Users\lazym\Documents\Code\dissertation\build\output\merged_fame_data.csv


In [ ]:
# Select from duckdb where any company doesn't have a registered number
con = ibis.duckdb.connect(str(db_path))
query = con.table("fame_id_data").filter(con.table("fame_id_data").registered_number.isnull() | (con.table("fame_id_data").registered_number == ""))
for row in query.execute().to_dict(orient="records"):
    print(row)
# Bind Ibis to the existing DuckDB tables
fixed = con.table("fame_fixed")
panel = con.table("fame_panel")
derived = con.table("fame_derived")

# Construct a lazy relational join using the registered_number key
master_query = (
    panel
    .left_join(fixed, "registered_number")
    .left_join(derived, "registered_number")
    # Explicitly select only the variables required for the current regression/analysis
    .select([
        panel.registered_number,
        panel.year,
        panel.turnover_gbp,
        fixed.primary_uk_sic_2007_code,
        derived.distance_to_parent_km
    ])
    # Apply global filters before execution to minimise memory load
    .filter(derived.active_subsidiary_flag == True)
)

# Execute the query in C++ and pull the final structured panel to Pandas
df_analysis = master_query.execute()